# Stateful 대화 (대화 기록 유지)

`1_6_stateless.ipynb`에서 보았듯이 LLM API는 **stateless**입니다.
요청 사이에 서버가 아무것도 기억하지 않기 때문에, 이전 대화를 기억하게 하려면
**매 요청마다 지금까지의 대화 전체를 messages에 다시 담아 보내야** 합니다.

이 노트북에서는 stateless를 stateful로 바꾸는 두 가지 방법을 봅니다.

1. `messages` 리스트에 **모델의 답변까지** 직접 누적하기
2. Gradio `ChatInterface`가 넘겨주는 `history`를 사용하기

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
openai = OpenAI()

In [2]:
system_prompt = """
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.
"""

## 1. messages에 직접 누적하기

핵심은 **assistant의 답변도 messages에 추가**하는 것입니다.
stateless 예제에서는 user 메시지만 보냈기 때문에 모델이 자기가 방금 한 말조차 몰랐습니다.

In [3]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "나의 이름은 이상화입니다."}
]

In [4]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
reply = response.choices[0].message.content

# 모델의 답변을 대화 기록에 추가 (이것이 stateful의 핵심)
messages.append({"role": "assistant", "content": reply})
print(reply)

이상화님, 반가워요! 😊  
처음 뵙는 거라서 그런지 더 궁금하네요.

지금 이상화님은 어떤 고민이나 이야기로 대화를 시작하고 싶으세요?  
편하게 말해 주세요—요즘 가장 신경 쓰이는 게 뭐예요?


In [5]:
# 다음 질문도 같은 messages 리스트에 이어서 추가합니다
messages.append({"role": "user", "content": "내 이름은 무엇인가요?"})

response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
reply = response.choices[0].message.content

messages.append({"role": "assistant", "content": reply})
print(reply)

이상화님이에요. 😊


In [6]:
# 지금까지 쌓인 대화 기록 확인
for m in messages:
    print(f"[{m['role']}] {m['content']}")

[system] 
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.

[user] 나의 이름은 이상화입니다.
[assistant] 이상화님, 반가워요! 😊  
처음 뵙는 거라서 그런지 더 궁금하네요.

지금 이상화님은 어떤 고민이나 이야기로 대화를 시작하고 싶으세요?  
편하게 말해 주세요—요즘 가장 신경 쓰이는 게 뭐예요?
[user] 내 이름은 무엇인가요?
[assistant] 이상화님이에요. 😊


## 2. 함수로 정리하기

매번 append를 직접 쓰는 대신 함수로 감싸면 편합니다.
`conversation` 리스트가 대화 상태(state)를 들고 있습니다.

In [7]:
conversation = [{"role": "system", "content": system_prompt}]

def ask(message, model="gpt-5.4-nano"):
    conversation.append({"role": "user", "content": message})
    response = openai.chat.completions.create(model=model, messages=conversation)
    reply = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": reply})
    return reply

def reset():
    """대화 기록을 초기화합니다."""
    global conversation
    conversation = [{"role": "system", "content": system_prompt}]

In [8]:
print(ask("나의 이름은 이상화입니다."))

이상화님, 만나서 반가워요! 😊  
요즘 어떤 일로 고민하고 계신가요, 아니면 그냥 편하게 이야기 나눠보고 싶으신가요?


In [9]:
print(ask("내 이름은 무엇인가요?"))

이상화님, 이름은 **이상화**예요. 😊  
앞에서 “나의 이름은 이상화입니다”라고 말씀해주셨어요.


In [10]:
print(f"대화 기록 개수: {len(conversation)}")

대화 기록 개수: 5


## 3. Gradio로 stateful 챗봇 만들기

`gr.ChatInterface`는 이전 대화를 `history` 인자로 넘겨줍니다.
`history`는 `[{"role": ..., "content": ...}, ...]` 형태로, OpenAI의 messages 형식과 같습니다.
그래서 변환 없이 messages에 바로 이어 붙일 수 있습니다.

stateless 버전과 비교해 보세요. 차이는 `+ history` 한 군데뿐입니다.

```python
# stateless
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": message}]

# stateful
messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
```

In [11]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content

In [12]:
dark_css = """
.gradio-container, .gradio-container .fillable, body {
    background-color: #000000 !important;
}
"""

with gr.Blocks() as chat_ui:
    gr.ChatInterface(fn=chat, title="With history")

chat_ui.launch(css=dark_css, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 4. 주의할 점

대화가 길어질수록 매 요청에 보내는 messages가 계속 커집니다.

- **비용/지연 증가**: 입력 토큰이 매 턴 누적됩니다.
- **컨텍스트 한계**: 모델의 최대 컨텍스트를 넘으면 오류가 납니다.

그래서 실제 서비스에서는 오래된 대화를 잘라내거나(truncation),
요약해서 보관하는(summarization) 방법을 씁니다.
이 내용은 `1_7_context_openai.ipynb`에서 이어집니다.